# Q4 — Machine Translation (Multi30k EN→DE)

**Canonical path**: Capped Seq2Seq+Attention vs Helsinki-NLP opus-mt comparison.
- Seq2Seq: 2000 train / 100 val / 100 test (trained from scratch)
- Transformer: 100 train / 100 val / 100 test (pretrained inference)

**Runtime**: T4 GPU required.

> This notebook reproduces the report-facing Q4 comparison artifacts.
> For uncapped Multi30k runs, see the **Exploratory** section at the bottom.

## 1. Environment Setup

In [ ]:
import os, sys, json, glob, shutil

try:
    from google.colab import drive
    drive.mount('/content/drive')
    ON_COLAB = True
except ImportError:
    ON_COLAB = False
    print('Not running on Colab — skipping Drive mount.')

DRIVE_OUTPUT = '/content/drive/MyDrive/467-takehome-outputs/q4' if ON_COLAB else None
if DRIVE_OUTPUT:
    os.makedirs(DRIVE_OUTPUT, exist_ok=True)

In [ ]:
REPO_DIR = '/content/467-takehome' if ON_COLAB else os.path.abspath('..')
if ON_COLAB:
    if not os.path.exists(REPO_DIR):
        !git clone https://github.com/zubeyralmaho/467-takehome.git {REPO_DIR}
    else:
        !cd {REPO_DIR} && git pull --ff-only
os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch
print(f'PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  GPU: {torch.cuda.get_device_name(0)}  |  VRAM: {torch.cuda.get_device_properties(0).total_mem/1e9:.1f} GB')

In [ ]:
def save_to_drive(run_dir, tag=''):
    if not DRIVE_OUTPUT:
        return
    name = os.path.basename(run_dir) + (f'_{tag}' if tag else '')
    dest = os.path.join(DRIVE_OUTPUT, name)
    shutil.copytree(run_dir, dest, dirs_exist_ok=True)
    print(f'Saved to Drive: {dest}')

def gpu_cleanup():
    import gc
    torch.cuda.empty_cache()
    gc.collect()
    if torch.cuda.is_available():
        print(f'GPU memory freed. Allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB')

## 2. Seq2Seq+Attention (GPU — Training)

Trained from scratch on 2000 train samples. Uses GRU encoder-decoder with attention.

In [ ]:
!python -m src.q4_machine_translation.main \
    --config configs/q4.yaml \
    --final-eval \
    --override \
        models.seq2seq.enabled=true \
        models.transformer.enabled=false \
        dataset.limit_train_samples=2000 \
        dataset.limit_validation_samples=100 \
        dataset.limit_test_samples=100 \
        models.seq2seq.max_epochs=8 \
        models.seq2seq.early_stopping_patience=2 \
        models.seq2seq.hidden_dim=128 \
        models.seq2seq.embedding_dim=128 \
        models.seq2seq.num_layers=1 \
        models.seq2seq.dropout=0.2 \
        models.seq2seq.batch_size=32 \
        models.seq2seq.learning_rate=0.001 \
        models.seq2seq.teacher_forcing_ratio=0.5

In [ ]:
seq2seq_run = sorted(glob.glob('outputs/q4/run_*'))[-1]
save_to_drive(seq2seq_run, 'seq2seq')
gpu_cleanup()

## 3. Helsinki-NLP Transformer (GPU — Pretrained Inference)

Pretrained opus-mt-en-de. Inference only, no training.

In [ ]:
!python -m src.q4_machine_translation.main \
    --config configs/q4.yaml \
    --final-eval \
    --override \
        models.seq2seq.enabled=false \
        models.transformer.enabled=true \
        dataset.limit_train_samples=100 \
        dataset.limit_validation_samples=100 \
        dataset.limit_test_samples=100 \
        models.transformer.batch_size=8

In [ ]:
transformer_run = sorted(glob.glob('outputs/q4/run_*'))[-1]
save_to_drive(transformer_run, 'transformer')
gpu_cleanup()

## 4. Generate Report Comparison Artifact

The Q4 report summary and figure are produced by dedicated scripts.

In [ ]:
print('Canonical Q4 runs:')
print(f'  Seq2Seq:     {seq2seq_run}')
print(f'  Transformer: {transformer_run}')
print()
print('To regenerate report artifacts:')
print('  python scripts/q4_report_summary.py')
print('  python scripts/report_comparison_figures.py')

## 5. Results Summary

In [ ]:
print('=== Q4 Canonical Results ===')
for label, run_dir in [('Seq2Seq', seq2seq_run), ('Transformer', transformer_run)]:
    metrics_file = os.path.join(run_dir, 'metrics.json')
    if os.path.exists(metrics_file):
        with open(metrics_file) as f:
            m = json.load(f)
        print(f'\n--- {label} ({os.path.basename(run_dir)}) ---')
        print(json.dumps(m, indent=2, default=str)[:2000])
    else:
        print(f'\n--- {label}: metrics.json not found ---')

---
## (Exploratory) Full Multi30k Runs

These runs use uncapped Multi30k and are **not** part of the canonical report comparison. Uncomment to run.

In [ ]:
# # --- Seq2Seq full Multi30k, bigger model ---
# !python -m src.q4_machine_translation.main \
#     --config configs/q4.yaml \
#     --final-eval \
#     --override \
#         models.seq2seq.enabled=true \
#         models.transformer.enabled=false \
#         models.seq2seq.max_epochs=20 \
#         models.seq2seq.early_stopping_patience=5 \
#         models.seq2seq.hidden_dim=256 \
#         models.seq2seq.embedding_dim=256 \
#         models.seq2seq.num_layers=2 \
#         models.seq2seq.dropout=0.3 \
#         models.seq2seq.batch_size=64 \
#         dataset.limit_train_samples=null \
#         dataset.limit_validation_samples=null \
#         dataset.limit_test_samples=null

# # --- Transformer full Multi30k ---
# !python -m src.q4_machine_translation.main \
#     --config configs/q4.yaml \
#     --final-eval \
#     --override \
#         models.transformer.enabled=true \
#         models.seq2seq.enabled=false \
#         models.transformer.batch_size=16 \
#         dataset.limit_train_samples=null \
#         dataset.limit_validation_samples=null \
#         dataset.limit_test_samples=null

# Q4 — Machine Translation (Multi30k EN→DE)
**Models:** Seq2Seq+Attention (GPU, trained) → Helsinki-NLP opus-mt (GPU, pretrained)

**Dataset:** Full Multi30k (~29K train)

**Runtime:** T4 GPU required

## 1. Environment Setup

In [ ]:
import os, subprocess, sys

from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUTPUT = '/content/drive/MyDrive/467-takehome-outputs/q4'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print('Drive mounted, output dir ready.')

In [ ]:
REPO_DIR = '/content/467-takehome'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/zubeyralmaho/467-takehome.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## 2. Run Seq2Seq+Attention (GPU — Training)

Full Multi30k dataset, increased epochs for decent BLEU.

In [ ]:
!python -m src.q4_machine_translation.main \
    --config configs/q4.yaml \
    --final-eval \
    --override \
        models.seq2seq.enabled=true \
        models.transformer.enabled=false \
        models.seq2seq.max_epochs=20 \
        models.seq2seq.early_stopping_patience=5 \
        models.seq2seq.hidden_dim=256 \
        models.seq2seq.embedding_dim=256 \
        models.seq2seq.num_layers=2 \
        models.seq2seq.dropout=0.3 \
        models.seq2seq.batch_size=64 \
        models.seq2seq.learning_rate=0.001 \
        models.seq2seq.teacher_forcing_ratio=0.5 \
        dataset.limit_train_samples=null \
        dataset.limit_validation_samples=null \
        dataset.limit_test_samples=null

In [ ]:
import glob, shutil
latest_run = sorted(glob.glob('outputs/q4/run_*'))[-1]
dest = os.path.join(DRIVE_OUTPUT, os.path.basename(latest_run) + '_seq2seq')
shutil.copytree(latest_run, dest, dirs_exist_ok=True)
print(f'Seq2Seq results saved to Drive: {dest}')

In [ ]:
# Free GPU memory before next model
import torch, gc
torch.cuda.empty_cache()
gc.collect()
print(f"GPU memory freed. Allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")

## 3. Run Helsinki-NLP Pretrained Transformer (GPU — Inference)

In [ ]:
!python -m src.q4_machine_translation.main \
    --config configs/q4.yaml \
    --final-eval \
    --override \
        models.seq2seq.enabled=false \
        models.transformer.enabled=true \
        models.transformer.batch_size=16 \
        dataset.limit_train_samples=null \
        dataset.limit_validation_samples=null \
        dataset.limit_test_samples=null

In [ ]:
latest_run = sorted(glob.glob('outputs/q4/run_*'))[-1]
dest = os.path.join(DRIVE_OUTPUT, os.path.basename(latest_run) + '_transformer')
shutil.copytree(latest_run, dest, dirs_exist_ok=True)
print(f'Transformer results saved to Drive: {dest}')

## 4. Results Summary

In [ ]:
import json

print('=== Q4 Results Summary ===')
for run_dir in sorted(glob.glob('outputs/q4/run_*')):
    metrics_file = os.path.join(run_dir, 'metrics.json')
    if os.path.exists(metrics_file):
        with open(metrics_file) as f:
            metrics = json.load(f)
        print(f'\n--- {os.path.basename(run_dir)} ---')
        print(json.dumps(metrics, indent=2, default=str)[:2000])